In [1]:
!pip install ultralytics opencv-python-headless matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.5 MB/s eta 0:00:00


In [2]:
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Load pre-trained YOLOv8 model (PyTorch-based)
model = YOLO('yolov8n.pt')  # nano model, fast inference

In [4]:
video_path = 'sports_video.mp4'  # Replace with your video filename
print(f"Using video: {video_path}")

Using video: sports_video.mp4


In [7]:
!pip install ultralytics

from google.colab import files
uploaded = files.upload()

video_path = list(uploaded.keys())[0]

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.predict(
    source=video_path,
    conf=0.5,
    save=True,
    show=False
)

for frame_results in results:
    for box in frame_results.boxes:
        cls = int(box.cls[0])
        if cls in [0, 32]:
            print(f"Detected {model.names[cls]} at {box.xyxy}")

Saving sports_video.mp4 to sports_video.mp4

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/367) /content/sports_video.mp4: 352x640 7 persons, 1 car, 69.9ms
video 1/1 (frame 2/367) /content/sports_video.mp4: 352x640 7 persons, 9.3ms
video 1/1 (frame 3/367) /content/sports_video.mp4: 352x640 7 persons, 1 sports ball, 16.6ms
video 1/1 (frame 4/367) /content/sports_video.mp4: 352x640 6 persons, 1 sports ball, 9.1ms
video 1/1 (frame 5/367) /content/sports_video.mp4: 352x640 6 persons, 1

**Filter Only Ball**

In [8]:
for frame_results in results:
    for box in frame_results.boxes:
        cls = int(box.cls[0])
        if cls == 32:  # Only ball
            print(f"Ball detected at {box.xyxy}")


Ball detected at tensor([[1348.8132,  952.9600, 1446.9945, 1036.0974]], device='cuda:0')
Ball detected at tensor([[1360.2371,  954.0318, 1453.2231, 1035.2805]], device='cuda:0')
Ball detected at tensor([[1374.7063,  951.2244, 1461.4327, 1037.1918]], device='cuda:0')
Ball detected at tensor([[1379.2238,  949.1234, 1474.4707, 1034.2794]], device='cuda:0')
Ball detected at tensor([[1399.3539,  948.3417, 1488.1743, 1034.4673]], device='cuda:0')
Ball detected at tensor([[1414.7935,  949.6393, 1504.2495, 1036.9106]], device='cuda:0')
Ball detected at tensor([[1423.7515,  948.0961, 1513.8141, 1039.8533]], device='cuda:0')
Ball detected at tensor([[1436.4254,  947.0193, 1528.9086, 1033.7041]], device='cuda:0')
Ball detected at tensor([[1454.4323,  943.0908, 1543.7806, 1033.3058]], device='cuda:0')
Ball detected at tensor([[1555.8091,  941.8863, 1633.3662, 1024.6332]], device='cuda:0')
Ball detected at tensor([[1561.2726,  939.2143, 1649.2025, 1027.2822]], device='cuda:0')
Ball detected at tens

**Display Names Objects Except Player and Ball**

In [9]:
for frame_results in results:
    detected_names = set()  # to avoid duplicates per frame
    for box in frame_results.boxes:
        cls = int(box.cls[0])
        if cls not in [0, 32]:  # exclude person and ball
            detected_names.add(model.names[cls])
    if detected_names:
        print(f"Objects detected in frame: {', '.join(detected_names)}")

Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: bench
Objects detected in frame: bench
Objects detected in frame: bench
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Objects detected in frame: car
Ob

**Draw bounding boxes on the video**

In [11]:
!pip install ultralytics opencv-python

from google.colab import files
uploaded = files.upload()

video_path = list(uploaded.keys())[0]

from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(video_path)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter("output.mp4", fourcc, fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=0.5)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])

            if cls in [0, 32]:
                label = model.names[cls]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()

from google.colab import files
files.download("output.mp4")

Saving sports_video.mp4 to sports_video (1).mp4

0: 352x640 7 persons, 1 car, 9.6ms
Speed: 2.6ms preprocess, 9.6ms inference, 1.3ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 7 persons, 9.9ms
Speed: 2.9ms preprocess, 9.9ms inference, 1.8ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 7 persons, 1 sports ball, 9.6ms
Speed: 2.8ms preprocess, 9.6ms inference, 1.8ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 6 persons, 1 sports ball, 11.0ms
Speed: 3.8ms preprocess, 11.0ms inference, 1.8ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 6 persons, 1 sports ball, 9.2ms
Speed: 3.9ms preprocess, 9.2ms inference, 1.8ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 6 persons, 1 sports ball, 9.5ms
Speed: 4.8ms preprocess, 9.5ms inference, 1.7ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 6 persons, 1 sports ball, 10.3ms
Speed: 5.0ms preprocess, 10.3ms inference, 1.8ms postprocess per image at shape (1, 3, 3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:

from ultralytics import YOLO

# Load pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # nano = fast, yolov8s = more accurate

# Run detection on the uploaded video
results = model.predict(source='sports_video.mp4', show=True, conf=0.5, save=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/367) /content/sports_video.mp4: 352x640 7 persons, 1 car, 7.4ms
video 1/1 (frame 2/367) /content/sports_video.mp4: 352x640 7 persons, 10.3ms
video 1/1 (frame 3/367) /content/sports_video.mp4: 352x640 7 persons, 1 sports ball, 10.5ms
video 1/1 (frame 4/367) /content/sports_video.mp4: 352x640 6 persons, 1 sports ball, 9.7ms
video 1/1 (frame 5/367) /content/sports_video.mp4: 352x640 6 persons, 1 sports ball, 9.5ms
video 1/1 (frame 6/367) 